# Son XAI sağlamlık analizi: LIME sadakati ve SHAP–LIME örneklem duyarlılığı

**Yayın öncesi tamamlayıcı deney (bağımsız çalışır).** Daha önce üretilen `kredi_ek_deney_sonuclari_paper.zip` **ve** `kredi_xai_sonuclari_paper.zip` dosyalarını kullanır. Yalnızca XGBoost ağırlıklı/ağırlıksız sürümlerini, ham/kalibre olasılıkları ve aynı dış CV bölmelerini değerlendirir.

**Önceden belirlenmiş üç LIME konfigürasyonu:** (1) `Baseline`: orijinal çeyreklik dilimleme + standart kernel; (2) `NarrowKernel`: aynı dilimleme + yarı kernel genişliği; (3) `Continuous`: sürekli değişkenleri dilimlemeden standart kernel. İlk beş özellik ve tam özellik sıralaması karşılaştırılır. **Birinci amaç**, LIME'ın *eğitim komşuluğundaki ağırlıklı R²* ile **bağımsız yeniden örneklenmiş komşulukta ağırlıklı R²** arasındaki farkı görmek; ikinci amaç eski/açıklanmamış dış test örneklerinin ve rassal tohumların etkisini ölçmek.

**Dikkat:** Yeni komşuluk örnekleri LIME'ın sentetik perturbasyon dağılımından gelir, gerçek kredi başvurularının doğal veya nedensel dağılımını temsil etmez. Burada ölçülen, yerel *surrogate fidelity* olup model doğruluğu/fairness değildir. `FAST` sonuçlarını makaleye koymayın. Kalibrasyon ve model tahminleri kaynak PAPER ZIP ile doğrulanmadan sonuç oluşturulmaz. Standart CPU Colab'da PAPER modu uzun sürebilir.

In [ ]:
%pip -q install "ucimlrepo>=0.0.7" "xgboost>=2.0,<4" "scikit-learn>=1.4,<2" "shap>=0.46,<1" "lime==0.2.0.1" "requests>=2.28"

## 1. Ayarlar ve önceki sonuç ZIP'leri

Önce `MODE='FAST'` ile test edin. Sorun çıkmazsa `PAPER` olarak tekrar çalıştırın. `PAPER`: her kat için önceki XAI çalışmasındaki 4 gözlem + daha önce XAI için seçilmemiş 4 gözlem; beş kat × iki veri × iki model × iki olasılık türü × üç LIME konfigürasyonu × iki tohum. Aynı gözlem için iki koşulda aynı rassal komşuluk kullanılır.

Çalışma veri setlerini UCI'den indireceği için internet gerekir. Önceki XAI ZIP dosyası yalnızca **eski gözlemlerin SHAP top-5 sonuçları** için kullanılır; yeni gözlemlerin SHAP açıklamaları tekrar hesaplanır.

In [ ]:
from pathlib import Path
import os, io, zipfile, json, time, warnings, itertools, inspect
import numpy as np, pandas as pd, requests, scipy, sklearn, xgboost, shap
from ucimlrepo import fetch_ucirepo
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import pairwise_distances
from scipy.stats import spearmanr
from lime.lime_tabular import LimeTabularExplainer
from IPython.display import display
warnings.filterwarnings('ignore', category=FutureWarning)

MODE = 'FAST'  # FAST tamamlandıktan sonra PAPER
assert MODE in ('FAST','PAPER')
SEED=42
LEGACY_PER_FOLD = 1 if MODE=='FAST' else 4
NEW_PER_FOLD = 1 if MODE=='FAST' else 4
LIME_REPEATS = 1 if MODE=='FAST' else 2
LIME_NUM_SAMPLES = 350 if MODE=='FAST' else 1500
HOLDOUT_SAMPLES = 200 if MODE=='FAST' else 700
SHAP_BACKGROUND_N = 15 if MODE=='FAST' else 35
SHAP_PERMUTATIONS = 2 if MODE=='FAST' else 6
BOOTSTRAPS = 300 if MODE=='FAST' else 2000
MODELS=('XGB_Unweighted','XGB_Weighted')
P_MODES=('raw','calibrated')
SETTINGS={
 'Baseline':      dict(discretize=True,kernel_scale=1.0),
 'NarrowKernel':  dict(discretize=True,kernel_scale=0.5),
 'Continuous':    dict(discretize=False,kernel_scale=1.0),
}

paper_path=Path('/content/kredi_ek_deney_sonuclari_paper.zip')
xai_path=Path('/content/kredi_xai_sonuclari_paper.zip')
needed=[p for p in (paper_path,xai_path) if not p.exists()]
if needed:
    try:
        from google.colab import files
        print('Lütfen iki PAPER ZIP dosyasını birden yükleyin.')
        uploaded=files.upload()
    except ImportError:
        raise FileNotFoundError('Colab dışında iki PAPER ZIP dosyasının yolunu tanımlayın.')
    for name in uploaded:
        if name=='kredi_ek_deney_sonuclari_paper.zip': paper_path=Path(name)
        if name=='kredi_xai_sonuclari_paper.zip': xai_path=Path(name)
assert paper_path.exists() and xai_path.exists(), 'İki PAPER ZIP dosyası da gerekli.'
with zipfile.ZipFile(paper_path) as z:
    main_meta=json.loads(z.read('manifest.json'))
    tuning=pd.read_csv(io.BytesIO(z.read('tuning.csv')))
    predictions=pd.read_csv(io.BytesIO(z.read('pooled_predictions.csv')))
with zipfile.ZipFile(xai_path) as z:
    prior_meta=json.loads(z.read('xai_manifest.json'))
    previous_xai=pd.read_csv(io.BytesIO(z.read('xai_cases.csv')))
assert main_meta['mode']=='PAPER' and main_meta['outer_folds']==5 and main_meta['random_state']==42
assert prior_meta['mode']=='PAPER' and prior_meta['n_cases']==320
assert set(MODELS).issubset(set(tuning.Model.unique()))
assert previous_xai.groupby(['Dataset','Fold','Model','Probability']).size().eq(8).all()
OUT=Path('/content/kredi_xai_final_saglamlik_'+MODE.lower() if Path('/content').exists() else '/mnt/data/kredi_xai_final_saglamlik_'+MODE.lower())
OUT.mkdir(parents=True,exist_ok=True)
print('Dosyalar doğrulandı:',len(predictions),'tahmin;',len(previous_xai),'önceki XAI kaydı')

## 2. Önceki deneyle aynı veri, kategorik kodlama, modeller ve katlar

In [ ]:
# UCI 573 için doğrudan ZIP içindeki SouthGermanCredit.asc dosyası
SG_DE=['laufkont','laufzeit','moral','verw','hoehe','sparkont','beszeit','rate','famges','buerge','wohnzeit','verm','alter','weitkred','wohn','bishkred','beruf','pers','telef','gastarb','kredit']
SG_EN=['status','duration','credit_history','purpose','amount','savings','employment_duration','installment_rate','personal_status_sex','other_debtors','present_residence','property','age','other_installment_plans','housing','number_credits','job','people_liable','telephone','foreign_worker','credit_risk']

def load_data():
    t=fetch_ucirepo(id=350)
    Xt=t.data.features.copy().reset_index(drop=True).drop(columns=['ID'],errors='ignore')
    yt=t.data.targets.iloc[:,0].astype(int).reset_index(drop=True)
    Xt.columns=['LIMIT_BAL','SEX','EDUCATION','MARRIAGE','AGE','PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6','BILL_AMT1','BILL_AMT2','BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6','PAY_AMT1','PAY_AMT2','PAY_AMT3','PAY_AMT4','PAY_AMT5','PAY_AMT6']
    url='https://archive.ics.uci.edu/static/public/573/south+german+credit+update.zip'
    response=requests.get(url,timeout=90); response.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        candidates=[v for v in z.namelist() if Path(v).name.lower()=='southgermancredit.asc']
        if len(candidates)!=1: raise RuntimeError('UCI ZIP dosyasında beklenen veri bulunmadı.')
        raw=z.read(candidates[0])
    Xy=pd.read_csv(io.BytesIO(raw),sep=r'\s+',engine='python')
    if Xy.shape==(999,21) and all(str(k).isdigit() for k in Xy.columns):
        Xy=pd.read_csv(io.BytesIO(raw),sep=r'\s+',engine='python',header=None,names=SG_DE)
    if list(Xy.columns)==SG_DE: Xy.columns=SG_EN
    if list(Xy.columns)!=SG_EN: raise ValueError('South German sütun sırası beklenmiyor: '+str(Xy.columns.tolist()))
    Xy=Xy.apply(pd.to_numeric,errors='raise')
    yg=(Xy.pop('credit_risk').astype(int)==0).astype(int).reset_index(drop=True)
    Xg=Xy.reset_index(drop=True)
    assert Xt.shape==(30000,23) and int(yt.sum())==6636
    assert Xg.shape==(1000,20) and int(yg.sum())==300
    return {'Taiwan_Credit_Default':(Xt,yt,['SEX','EDUCATION','MARRIAGE']),
            'South_German_Credit':(Xg,yg,[c for c in Xg.columns if c not in ('duration','amount','age')])}

def make_prep(X,cats):
    num=[c for c in X.columns if c not in cats]
    return ColumnTransformer([
        ('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),
        ('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),cats)
    ],sparse_threshold=0.)

def get_xgb(X,cats,y,weighted,params):
    args=dict(objective='binary:logistic',eval_metric='logloss',tree_method='hist',n_jobs=1,random_state=42)
    if weighted: args['scale_pos_weight']=float((y==0).sum()/max(1,(y==1).sum()))
    model=Pipeline([('pre',make_prep(X,cats)),('clf',xgboost.XGBClassifier(**args))])
    return model.set_params(**json.loads(params))

def select_ids(y,ids,n,seed):
    rng=np.random.default_rng(seed)
    pos=[i for i in ids if int(y.iloc[i])==1]; neg=[i for i in ids if int(y.iloc[i])==0]
    take_pos=min(len(pos),n//2); take_neg=min(len(neg),n-take_pos)
    selected=list(rng.choice(pos,size=take_pos,replace=False))+list(rng.choice(neg,size=take_neg,replace=False))
    if len(selected)<n:
        rest=np.setdiff1d(ids,np.asarray(selected,dtype=int))
        selected+=list(rng.choice(rest,size=min(n-len(selected),len(rest)),replace=False))
    return sorted(map(int,selected))

datasets=load_data()
for ds,(X,y,cats) in datasets.items(): print(ds,'shape=',X.shape,'positive=',int(y.sum()))

## 3. Açıklama ayarları ve bağımsız komşulukta LIME sadakati

LIME'ın `exp.score` çıktısı **öğretildiği perturbasyonlar üzerindeki ağırlıklı R²**'dir. Eklediğimiz `Holdout_Weighted_R2`, aynı LIME surrogate'ını farklı perturbasyonlarda test eder. LIME 0.2.0.1 içinde `__data_inverse` özel yordamı gereklidir; yordam beklendiği gibi çalışmazsa **analiz hata vererek durur**, sessizce değişmiş metrik üretmez. <https://github.com/marcotcr/lime/blob/master/lime/lime_tabular.py>

Sonuçlar üç ayar için *aynı sayıdaki özellik* ve *aynı perturbasyon sayısı* kullanılarak hesaplanır; tek değişiklik kernel genişliği veya sürekli özelliklerin dilimlenmesidir.
**Sürüm uyumluluğu düzeltmesi:** Kurulu LIME sürümlerinde `explain_instance` ve özel `__data_inverse` yordamlarının `sampling_method` parametre imzaları farklı olabilir. Kod parametreyi yalnızca fonksiyon kabul ediyorsa geçirir; diğer durumda varsayılan Gaussian örnekleme kullanılır. Özel yordamın kullanımının sürüm-bağımlı olduğu metodolojik sınırlılık devam eder.


In [ ]:
def predict_fn(pipeline,cols):
    def f(z):
        z=np.asarray(z,dtype=float)
        if z.ndim==1: z=z.reshape(1,-1)
        return pipeline.predict_proba(pd.DataFrame(z,columns=cols))[:,1]
    return f

def make_lime(Xtr,cols,cats,setting,seed):
    kw=np.sqrt(len(cols))*0.75*setting['kernel_scale']
    return LimeTabularExplainer(Xtr[cols].to_numpy(dtype=float),mode='classification',
       feature_names=cols,class_names=['NonDefault','Default'],
       categorical_features=[cols.index(c) for c in cats],
       discretize_continuous=setting['discretize'],kernel_width=kw,
       feature_selection='none',random_state=seed)

def topk(v,k=5):
    return set(np.argsort(-np.abs(np.asarray(v)),kind='stable')[:k].tolist())

def jac(a,b):
    a,b=set(a),set(b)
    return float(len(a&b)/len(a|b)) if (a|b) else 1.

def rank_rho(a,b):
    a,b=np.abs(np.asarray(a,dtype=float)),np.abs(np.asarray(b,dtype=float))
    return float(spearmanr(a,b).statistic) if a.std()>1e-10 and b.std()>1e-10 else np.nan

def lime_fit_and_holdout(explainer,x,positive_fn,num_samples,holdout_n):
    def positive_two(z):
        p=np.clip(positive_fn(z),0.,1.)
        return np.column_stack((1.-p,p))
    # LIME versions differ: some accept sampling_method, others always use
    # their default Gaussian neighborhood. Check the installed signature.
    explain_kwargs=dict(labels=(1,),num_features=len(x),num_samples=num_samples)
    if 'sampling_method' in inspect.signature(explainer.explain_instance).parameters:
        explain_kwargs['sampling_method']='gaussian'
    exp=explainer.explain_instance(np.asarray(x,dtype=float),positive_two,
                                   **explain_kwargs)
    coef=np.zeros(len(x),dtype=float)
    for j,value in exp.as_map()[1]:coef[int(j)]=float(value)
    # This protected API is the same perturbation generator used inside LIME 0.2.0.1.
    generate_holdout=explainer._LimeTabularExplainer__data_inverse
    holdout_args=(np.asarray(x,dtype=float),holdout_n)
    if 'sampling_method' in inspect.signature(generate_holdout).parameters:
        val_data,val_inverse=generate_holdout(*holdout_args,sampling_method='gaussian')
    else:
        val_data,val_inverse=generate_holdout(*holdout_args)
    val_scaled=(np.asarray(val_data,dtype=float)-explainer.scaler.mean_)/explainer.scaler.scale_
    val_dist=pairwise_distances(val_scaled,val_scaled[:1],metric='euclidean').ravel()
    val_w=np.asarray(explainer.base.kernel_fn(val_dist),dtype=float)
    true_y=np.asarray(positive_fn(val_inverse),dtype=float)
    pred_y=float(exp.intercept[1])+np.asarray(val_scaled@coef,dtype=float)
    # Surrogate at the instance must match LIME's own local_pred; otherwise reject the metric.
    loc_ref=float(np.asarray(exp.local_pred).reshape(-1)[0]);loc_here=float(pred_y[0])
    if not np.isfinite(loc_here) or abs(loc_ref-loc_here)>1e-5:
        raise RuntimeError(f'LIME surrogate projection is inconsistent: {loc_ref=} {loc_here=}')
    wavg=float(np.average(true_y,weights=val_w))
    denominator=float(np.sum(val_w*(true_y-wavg)**2))
    external_r2=(1.-float(np.sum(val_w*(true_y-pred_y)**2))/denominator) if denominator>1e-12 else np.nan
    return coef,float(exp.score),float(external_r2),loc_ref

def choose_legacy_new(y,outer_ids,previous_ids,n_old,n_new,seed):
    rng=np.random.default_rng(seed)
    old=sorted(int(v) for v in previous_ids)
    assert len(old)==8
    # Balanced choices for both cohorts using target y, subject to available candidates.
    def take_balanced(pool,n):
        positives=[i for i in pool if int(y.iloc[i])==1]
        negatives=[i for i in pool if int(y.iloc[i])==0]
        a=min(len(positives),n//2); b=min(len(negatives),n-a)
        chosen=list(rng.choice(positives,a,replace=False))+list(rng.choice(negatives,b,replace=False))
        if len(chosen)<n:
            rest=list(set(pool)-set(map(int,chosen)))
            chosen+=list(rng.choice(rest,n-len(chosen),replace=False))
        return sorted(map(int,chosen))
    a=take_balanced(old,n_old)
    fresh=sorted(set(map(int,outer_ids))-set(old))
    b=take_balanced(fresh,n_new)
    assert set(a).isdisjoint(b)
    return a,b

assert jac({1,2,3},{2,3,4})==0.5
assert topk([1,5,3],2)=={1,2}
print('LIME explain_instance sampling_method desteği:',
      'sampling_method' in inspect.signature(LimeTabularExplainer.explain_instance).parameters)
print('LIME __data_inverse sampling_method desteği:',
      'sampling_method' in inspect.signature(LimeTabularExplainer._LimeTabularExplainer__data_inverse).parameters)
print('Açıklama ve örneklem yardımcı fonksiyonları hazır.')

## 4. Ana deney (uzun sürecek)

Önceki dış test tahminleriyle her modelin ham ve kalibre olasılığı birebir uyuşmak zorundadır. Eski gözlemlerin SHAP top-5 sonuçları kaynak ZIP dosyasından alınır, yeni gözlemlerin SHAP top-5 özellikleri aynı SHAP-permutation ayarıyla hesaplanır.

Her fold sonrasında `lime_case_repeats.csv` otomatik kaydedilir. Bir kesinti olursa mevcut tablo üzerinden son başarılı kat görülebilir; çalışma yeniden çalıştırıldığında baştan üretir. `PAPER` sırasında hata alırsanız hata mesajını gönderin, eksik çıktıları makaleye eklemeyin.

In [ ]:
rows=[]; repro_rows=[]; began=time.time()
previous_lookup=previous_xai.set_index(['Dataset','Fold','Model','Probability','RowID'])
for ds_name,(X,y,cats) in datasets.items():
    cols=X.columns.tolist(); folds=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
    for fold,(itr,ite) in enumerate(folds.split(X,y),1):
        Xtr,ytr=X.iloc[itr],y.iloc[itr]
        old_ids=(previous_xai[(previous_xai.Dataset==ds_name)&(previous_xai.Fold==fold)&
                    (previous_xai.Model=='XGB_Unweighted')&(previous_xai.Probability=='raw')].RowID.tolist())
        legacy,new=choose_legacy_new(y,ite,old_ids,LEGACY_PER_FOLD,NEW_PER_FOLD,2026+fold)
        subjects=[('legacy',i) for i in legacy]+[('new',i) for i in new]
        bg=Xtr.sample(n=min(SHAP_BACKGROUND_N,len(Xtr)),random_state=SEED+fold)[cols].to_numpy(dtype=float)
        # Construct all 3 settings just once per CV fold. Random states are reset per case.
        expl_by_setting={name:make_lime(Xtr,cols,cats,cfg,SEED+fold)
                         for name,cfg in SETTINGS.items()}
        for model_name in MODELS:
            sel=tuning[(tuning.Dataset==ds_name)&(tuning.Fold==fold)&(tuning.Model==model_name)]
            assert len(sel)==1
            trained=get_xgb(Xtr,cats,ytr,model_name=='XGB_Weighted',sel.iloc[0].Best_Params)
            trained.fit(Xtr,ytr)
            cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=SEED+fold)
            calibrated=CalibratedClassifierCV(estimator=clone(trained),method='sigmoid',cv=cv,ensemble=False)
            calibrated.fit(Xtr,ytr)
            for probability,pipeline in [('raw',trained),('calibrated',calibrated)]:
                prev=(predictions[(predictions.Dataset==ds_name)&(predictions.Fold==fold)&
                       (predictions.Model==model_name)&(predictions.Probability==probability)]
                     .set_index('RowID'))
                actual=pipeline.predict_proba(X.iloc[ite])[:,1]
                max_error=float(np.max(np.abs(pd.Series(actual,index=ite).loc[prev.index].to_numpy()-prev.P.to_numpy())))
                repro_rows.append(dict(Dataset=ds_name,Fold=fold,Model=model_name,Probability=probability,
                                       Max_Error=max_error))
                if max_error>1e-3: raise RuntimeError(f'Model repro failed: {ds_name} {fold} {model_name} {probability} {max_error}')
                fn=predict_fn(pipeline,cols)
                needed_new=np.array(new,dtype=int)
                fresh_top={}
                if len(needed_new):
                    masker=shap.maskers.Independent(bg,max_samples=len(bg))
                    shap_xpl=shap.PermutationExplainer(fn,masker,seed=SEED+fold)
                    for row_id in needed_new:
                        xx=X.iloc[row_id][cols].to_numpy(dtype=float)
                        sh=shap_xpl(xx.reshape(1,-1),max_evals=SHAP_PERMUTATIONS*2*len(cols)+1,
                                     batch_size=200,silent=True)
                        sv=np.asarray(sh.values[0],dtype=float).reshape(-1)
                        fresh_top[int(row_id)]=topk(sv)
                for cohort,row_id in subjects:
                    x=X.iloc[row_id][cols].to_numpy(dtype=float)
                    if cohort=='legacy':
                        old=previous_lookup.loc[(ds_name,fold,model_name,probability,int(row_id))]
                        assert abs(float(old.Prediction)-float(fn(x)[0]))<1e-3
                        shap_t={cols.index(name) for name in json.loads(old.SHAP_Top5)}
                    else:shap_t=fresh_top[int(row_id)]
                    for setting_name,lime_exp in expl_by_setting.items():
                        for repeat in range(LIME_REPEATS):
                            # Paired perturbation seeds for raw/calibrated and weighted/unweighted.
                            case_seed=int((SEED+1009*fold+4999*int(row_id)+131*repeat+
                                           17*list(SETTINGS).index(setting_name))%(2**32-1))
                            lime_exp.random_state.seed(case_seed)
                            co,train_r2,oos_r2,local_surrogate=lime_fit_and_holdout(
                                lime_exp,x,fn,LIME_NUM_SAMPLES,HOLDOUT_SAMPLES)
                            rows.append(dict(Dataset=ds_name,Fold=fold,Model=model_name,
                                Probability=probability,RowID=int(row_id),Cohort=cohort,
                                Config=setting_name,Repeat=repeat,Seed=case_seed,Y=int(y.iloc[row_id]),
                                Prediction=float(fn(x)[0]),InSample_R2=train_r2,Holdout_Weighted_R2=oos_r2,
                                Jaccard5=jac(shap_t,topk(co)),LIME_Surrogate_Local_Pred=local_surrogate,
                                LIME_Top5=json.dumps([cols[j] for j in np.argsort(-np.abs(co))[:5]]),
                                SHAP_Top5=json.dumps([cols[j] for j in sorted(shap_t)]),
                                LIME_Coefficients=json.dumps(co.tolist())))
                print(ds_name,'fold',fold,model_name,probability,'done',len(subjects),'cases',flush=True)
        pd.DataFrame(rows).to_csv(OUT/'lime_case_repeats.csv',index=False)
        pd.DataFrame(repro_rows).to_csv(OUT/'prediction_reproducibility.csv',index=False)
print('Tüm vaka düzeyi deneyler tamamlandı. Dakika:',round((time.time()-began)/60,1))

## 5. Birim kontrolleri, eşleştirilmiş bootstrap ve makale tabloları

Her gözlemin iki seed'den gelen skorunun **önce ortalaması** alınır; seed'ler bağımsız müşteri gibi çoğaltılmaz. Eşleştirilmiş bootstrap, dış CV eğitim belirsizliğini kapsamaz. Farklarda pozitif değer, **ilk yazılan koşulun skorunun daha yüksek** olduğunu gösterir. Bootstrap yüzdelikleri çoklu test için doğrulanmış istatistiksel anlamlılık olarak kullanılmamalıdır.

In [ ]:
res=pd.read_csv(OUT/'lime_case_repeats.csv')
rep=pd.read_csv(OUT/'prediction_reproducibility.csv')
assert rep.Max_Error.max()<1e-3
required=['Dataset','Fold','Model','Probability','RowID','Cohort','Config']
assert res.groupby(required).size().eq(LIME_REPEATS).all()
expected=2*5*2*2*len(SETTINGS)*(LEGACY_PER_FOLD+NEW_PER_FOLD)*LIME_REPEATS
assert len(res)==expected,(len(res),expected)
assert not res.duplicated(required+['Repeat']).any()
assert set(res.Config)==set(SETTINGS)
assert (res.InSample_R2<=1.+1e-8).all()
assert np.isfinite(res.Holdout_Weighted_R2).all(), 'Yakın komşulukta OOS ağırlıklı R² hesaplanamadı.'

CASE_KEYS=['Dataset','Fold','Model','Probability','RowID','Cohort','Config']
means=(res.groupby(CASE_KEYS,as_index=False)
         .agg(InSample_R2=('InSample_R2','mean'),Holdout_Weighted_R2=('Holdout_Weighted_R2','mean'),
              Jaccard5=('Jaccard5','mean')))
means['Overfit_Gap']=means.InSample_R2-means.Holdout_Weighted_R2
summary=(means.groupby(['Dataset','Model','Probability','Config'],as_index=False)
 .agg(N=('RowID','size'),InSample_R2_Mean=('InSample_R2','mean'),
      Holdout_Weighted_R2_Mean=('Holdout_Weighted_R2','mean'),
      Holdout_Weighted_R2_Median=('Holdout_Weighted_R2','median'),
      Overfit_Gap_Mean=('Overfit_Gap','mean'),Jaccard5_Mean=('Jaccard5','mean')))
summary.to_csv(OUT/'summary_by_config.csv',index=False)

cohort_summary=(means.groupby(['Dataset','Model','Probability','Config','Cohort'],as_index=False)
 .agg(N=('RowID','size'),Holdout_Weighted_R2_Mean=('Holdout_Weighted_R2','mean'),
      InSample_R2_Mean=('InSample_R2','mean'),Jaccard5_Mean=('Jaccard5','mean')))
cohort_summary.to_csv(OUT/'summary_by_cohort.csv',index=False)

def paired_interval(a,b,nb=1000,seed=2026):
    a,b=np.asarray(a,dtype=float),np.asarray(b,dtype=float)
    assert a.shape==b.shape and len(a)>=2
    d=a-b; rng=np.random.default_rng(seed)
    boot=np.empty(nb)
    for k in range(nb):
        ix=rng.integers(0,len(d),len(d));boot[k]=np.mean(d[ix])
    return float(d.mean()),float(np.quantile(boot,.025)),float(np.quantile(boot,.975))

paired=[]
for (ds,m,pm),tab in means.groupby(['Dataset','Model','Probability']):
    p=tab.pivot(index='RowID',columns='Config',values=['Holdout_Weighted_R2','Jaccard5']).dropna()
    for alternative in ['NarrowKernel','Continuous']:
        for metric in ('Holdout_Weighted_R2','Jaccard5'):
            delta,lo,hi=paired_interval(p[(metric,'Baseline')],p[(metric,alternative)],BOOTSTRAPS)
            paired.append(dict(Dataset=ds,Model=m,Probability=pm,Metric=metric,
                 Comparison='Baseline_minus_'+alternative,N=len(p),Diff=delta,CI_Lower=lo,CI_Upper=hi))
pd.DataFrame(paired).to_csv(OUT/'paired_config_comparisons.csv',index=False)

calcomp=[]
for (ds,m,setting),tab in means.groupby(['Dataset','Model','Config']):
    p=tab.pivot(index='RowID',columns='Probability',values=['Holdout_Weighted_R2','Jaccard5']).dropna()
    for metric in ('Holdout_Weighted_R2','Jaccard5'):
        delta,lo,hi=paired_interval(p[(metric,'raw')],p[(metric,'calibrated')],BOOTSTRAPS)
        calcomp.append(dict(Dataset=ds,Model=m,Config=setting,Metric=metric,
            Comparison='Raw_minus_Calibrated',N=len(p),Diff=delta,CI_Lower=lo,CI_Upper=hi))
pd.DataFrame(calcomp).to_csv(OUT/'paired_calibration_comparisons.csv',index=False)

stab=[]
for key,sub in res.groupby(CASE_KEYS):
    if len(sub)>=2:
        cc=[np.asarray(json.loads(s),dtype=float) for s in sub.LIME_Coefficients]
        pairs=list(itertools.combinations(cc,2))
        stab.append(dict(zip(CASE_KEYS,key),
            Seed_Jaccard5=float(np.mean([jac(topk(a),topk(b)) for a,b in pairs])),
            Seed_SpearmanAbs=float(np.nanmean([rank_rho(a,b) for a,b in pairs]))))
if stab:
    stability=pd.DataFrame(stab)
    stability.to_csv(OUT/'lime_seed_stability.csv',index=False)
    stability_summary=stability.groupby(['Dataset','Model','Probability','Config'],as_index=False).agg(
        N=('RowID','size'),Seed_Jaccard5_Mean=('Seed_Jaccard5','mean'),Seed_SpearmanAbs_Mean=('Seed_SpearmanAbs','mean'))
    stability_summary.to_csv(OUT/'lime_seed_stability_summary.csv',index=False)

print('KONTROLLER BAŞARILI, satırlar:',len(res),' vaka ortalamaları:',len(means))
display(summary.round(4))
display(pd.DataFrame(paired).round(4))
display(cohort_summary.round(4))

## 6. Sonuçları indirme

Oluşan tek ZIP dosyasını ChatGPT'ye gönderin. Revize makaleye *hangi* sağlamlık sonuçlarının ekleneceğini birlikte karar vereceğiz. Düşük held-out R² saptanırsa, SHAP–LIME uyumunu karar açıklamasının güvenilirliği olarak değil, iki yöntemin göreli örtüşmesine ilişkin sınırlı tanımlayıcı bulgu olarak sunacağız.

In [ ]:
meta={
 'study':'explanation_fidelity_and_sample_sensitivity', 'mode':MODE,
 'settings':SETTINGS,'legacy_per_fold':LEGACY_PER_FOLD,'new_per_fold':NEW_PER_FOLD,
 'lime_repeats':LIME_REPEATS,'lime_num_samples':LIME_NUM_SAMPLES,
 'holdout_samples':HOLDOUT_SAMPLES,'shap_background':SHAP_BACKGROUND_N,
 'shap_permutations':SHAP_PERMUTATIONS,'bootstrap_replicates':BOOTSTRAPS,
 'case_repeats':int(len(res)),'unique_explained_cases':int(len(means)),
 'reproducibility_max_error':float(rep.Max_Error.max()),
 'versions':{'numpy':np.__version__,'pandas':pd.__version__,'sklearn':sklearn.__version__,
             'shap':shap.__version__,'xgboost':xgboost.__version__,
             'lime':__import__('importlib.metadata',fromlist=['version']).version('lime')},
 'notes':['Within-fold predictions are reproduced exactly before explaining.',
          'Seed repeats are averaged before bootstrap; no pseudo-replication.',
          'Held-out neighborhood R2 measures faithfulness on independent LIME perturbations only.',
          'Bootstrap conditions on fixed trained models, and does not cover cross-validation model training variability.',
          'Original and new cases are stratified sampled, not claims about arbitrary population mixtures.']
}
(OUT/'robustness_manifest.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
file=Path('/content/kredi_xai_saglamlik_sonuclari_'+MODE.lower()+'.zip' if Path('/content').exists() else '/mnt/data/kredi_xai_saglamlik_sonuclari_'+MODE.lower()+'.zip')
with zipfile.ZipFile(file,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.glob('*.csv')):z.write(p,arcname=p.name)
    z.write(OUT/'robustness_manifest.json',arcname='robustness_manifest.json')
print('Dosya hazır:',file)
try:
    from google.colab import files
    files.download(str(file))
except ImportError:
    print('Yerel çalıştırma: çıktı dosyası',file)